In [ ]:
"""
Plik: train_yolo_colab.ipynb
Opis: Trening modelu YOLOv8n do detekcji obiektow (ball, player, referee)
      na obrazach z meczu pilki noznej. Srodowisko docelowe: Google Colab z GPU.
      Dane wejsciowe: archiwum w formacie YOLO przechowywane na Google Drive.
"""

# ==========================================================================
# 1. Konfiguracja srodowiska
# ==========================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

import os, shutil, glob, torch
from ultralytics import YOLO

# Weryfikacja dostepnego akceleratora
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

assert gpu_mem > 15, "Niewystarczajaca pamiec GPU - wymagane co najmniej 16 GB VRAM"

# ==========================================================================
# 2. Ekstrakcja danych z Google Drive na dysk lokalny VM
# ==========================================================================
# Rozpakowanie na dysk lokalny eliminuje narzut sieciowy podczas treningu.
zip_path = "/content/drive/MyDrive/CvFootballTracker_Data/Detection/yoloformat_large.zip"
local_data_dir = "/content/yoloformat"

if not os.path.exists(local_data_dir):
    print("Rozpakowywanie archiwum...")
    shutil.unpack_archive(zip_path, extract_dir="/content/")
    print("Zakonczono.")
else:
    print("Dane juz dostepne na dysku lokalnym.")

n_train = len(glob.glob(f"{local_data_dir}/train/*/images/*.jpg"))
n_valid = len(glob.glob(f"{local_data_dir}/valid/*/images/*.jpg"))
print(f"Liczba obrazow - train: {n_train}, valid: {n_valid}")

# ==========================================================================
# 3. Generacja pliku konfiguracyjnego data.yaml
# ==========================================================================
# Sciezki w standardzie POSIX (Linux), niezalezne od srodowiska zrodlowego.
yaml_content = f"""
path: {local_data_dir}
train: train
val: valid
test: test

names:
  0: ball
  1: player
  2: referee
"""
with open(f"{local_data_dir}/data.yaml", 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())

# ==========================================================================
# 4. Inicjalizacja modelu z obsluga wznawiania
# ==========================================================================
# Jezeli wczesniejszy trening zostal przerwany (np. timeout sesji),
# automatyczne podjecie z ostatniego checkpointu.
RUN_NAME    = "yolo_soccernet_n_v1"
PROJECT_DIR = "/content/drive/MyDrive/CvFootballTracker_Data/results"

last_ckpt = f"{PROJECT_DIR}/{RUN_NAME}/weights/last.pt"
if os.path.exists(last_ckpt):
    print(f"Wykryto checkpoint, wznawianie z: {last_ckpt}")
    model = YOLO(last_ckpt)
    resume = True
else:
    print("Inicjalizacja od wag pretrenowanych na zbiorze COCO")
    model = YOLO("yolov8n.pt")
    resume = False

# ==========================================================================
# 5. Trening
# ==========================================================================
results = model.train(
    # Dane wejsciowe
    data=f"{local_data_dir}/data.yaml",

    # Rozdzielczosc i wielkosc batcha
    imgsz=960,              # wyzsza rozdzielczosc poprawia detekcje malych obiektow
    batch=32,

    # Czas trwania treningu
    epochs=100,
    patience=30,            # early stopping przy braku poprawy mAP50 przez N epok

    # Harmonogram learning rate
    cos_lr=True,            # cosine annealing
    warmup_epochs=3,

    # Wydajnosc
    cache='ram',            # cache obrazow w RAM (wymaga High-RAM runtime)
    workers=8,
    amp=True,               # automatic mixed precision (FP16)
    device=0,

    # Parametry augmentacji
    mosaic=1.0,
    close_mosaic=15,        # wylaczenie mosaic w koncowych epokach dla stabilizacji
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,            # brak rotacji - stala perspektywa kamery boiska
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,             # bez odbicia pionowego
    fliplr=0.5,             # odbicie poziome - boisko symetryczne

    # Zapis wynikow
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save_period=-1,         # zapis wylacznie best.pt i last.pt
    resume=resume,

    # Reprodukowalnosc
    seed=17,
    deterministic=True,

    # Logowanie
    plots=True,
    verbose=True,
)

# ==========================================================================
# 6. Walidacja koncowa i raport metryk per klasa
# ==========================================================================
print("\n" + "="*60)
print("Walidacja koncowa (best.pt)")
print("="*60)

best_path = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
best_model = YOLO(best_path)
metrics = best_model.val(data=f"{local_data_dir}/data.yaml", split='val', imgsz=960)

print(f"\nmAP50 ogolne:    {metrics.box.map50:.4f}")
print(f"mAP50-95 ogolne: {metrics.box.map:.4f}")
print(f"\nmAP50-95 per klasa:")
for i, name in enumerate(['ball', 'player', 'referee']):
    print(f"  {name:10s} {metrics.box.maps[i]:.4f}")

print(f"\nSciezka do najlepszego modelu: {best_path}")